# Explore recorded benchmark results

Each recorded run has GitHub-readable Markdown, SVG plots, and raw JSON/CSV.
This notebook renders the exact published chart specifications with the same
Matplotlib module used by the exporter. It never combines runs or scenarios.
Set `AMBER_BENCH_RESULTS` and optionally `AMBER_BENCH_RUN` to select results.


In [ ]:
import json
import os
import sys
from pathlib import Path
from IPython.display import SVG, display

RESULTS = Path(os.environ.get("AMBER_BENCH_RESULTS", "results")).resolve()
if not RESULTS.is_dir() and Path.cwd().name == "results":
    RESULTS = Path.cwd()
REPO = RESULTS.parent
# When exploring results exported elsewhere, set AMBER_BENCH_REPO to this checkout.
REPO = Path(os.environ.get("AMBER_BENCH_REPO", str(REPO))).resolve()
sys.path.insert(0, str(REPO / "python"))
from amber_bench_plot import render_svg, validate_report

runs = sorted(RESULTS.glob("*/run.json"))
if not runs:
    raise FileNotFoundError("No recorded runs. Export a completed run first.")
for path in runs:
    entry = json.loads(path.read_text())
    print(entry["id"], entry["profile"], "repeats:", entry["repeats"], "valid:", entry["valid"])


In [ ]:
RUN_ID = os.environ.get("AMBER_BENCH_RUN", runs[-1].parent.name)
RUN = RESULTS / RUN_ID
report = json.loads((RUN / "report.json").read_text())
validate_report(report)
print("Profile:", report["profile"]["name"], "repetitions:", report["repeats"])
print("Cache policy:", report["host"]["cache_policy"])
print("Benchmark commit:", report.get("harness_commit"))
for name, tool in report["tools"]["tools"].items():
    if name in ("amber-go", "amber-rust"):
        print(name, tool.get("source_commit"), tool["sha256"])


## Scenario plots

Choose one scenario below. Bars show recorded medians; ticks show recorded p95
where available. Small sample counts do not support precise percentile estimates.
Unsupported operations remain labels, never zero-valued measurements.


In [ ]:
spec_paths = sorted((RUN / "plots").glob("*.json"))
if not spec_paths:
    raise ValueError("No chart specifications. Export with the current publisher.")
specs = [(p, json.loads(p.read_text())) for p in spec_paths]
scenarios = sorted({s["title"].split(" — ", 1)[0] for _, s in specs})
print("Scenarios:", scenarios)
SCENARIO = scenarios[0]  # Change this value to inspect another workload.


In [ ]:
for path, spec in specs:
    if spec["title"].split(" — ", 1)[0] == SCENARIO:
        display(SVG(render_svg(spec)))


## Raw samples

The raw tables preserve individual repetitions. Setup and verification are separate
from measurements. Review backend semantics before interpreting differences.


In [ ]:
import pandas as pd
samples = pd.read_csv(RUN / "samples.csv")
display(samples[samples["scenario"] == SCENARIO])
